# FADA fine-tune on Quran and hadith (question -> passage)
Run top to bottom. Upload `corpus.jsonl` first (made on your machine with `python scripts/quran_pairs.py --out data/quran/corpus.jsonl --hadith`; without `--hadith` the notebook trains on the Quran only).
The slow steps (question generation, negative mining, training) write to `cfg.out` and resume when rerun. Set `cfg.dry_run = True` for a 10-minute smoke test.

Data and licences: Tanzil text CC BY 3.0 (unchanged, attribution kept), Tafsir Center tafsir CC BY 4.0, AQQAC (`nazimali/quran-question-answer-context`) CC BY 4.0 used for training, **QRCD is CC BY-ND, so it is used only as the held-out test set, never for training**. Questions written by Qwen3 are synthetic. Hadith pairs come from chapter titles (title -> hadith matn) and from synthetic questions; no hadith relevance labels exist in the repo, so hadith is tested on whole **held-out chapters** (query = chapter title, gold = the chapter's hadiths).

In [ ]:
!pip install -q "sentence-transformers==6.1.0" "transformers==5.18.0" datasets accelerate lion-pytorch wandb "mteb==2.22.0" vllm

In [ ]:
import os, gc, re, json, random, hashlib
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# "" = load FADA from the W&B artifact below. Others: "hf:user/repo" | "wandb:entity/project/artifact:alias" | "path:/content/..."
FADA_SOURCE = ""

@dataclass
class Cfg:
    fada_source: str = FADA_SOURCE
    fada_artifact: str = "arbertv2-mrl-embeddings/arbertv2-mrl-final:latest"
    corpus_path: str = "/content/corpus.jsonl"           # upload it (Files panel), made by scripts/quran_pairs.py
    generator: str = "Qwen/Qwen3-32B"                    # bf16 ~64 GB; use "Qwen/Qwen3-8B" if memory is short
    questions_per_item: int = 5
    synthetic_ayat_sources: tuple = ("quran",)            # sources that get generated questions (their ayat)
    synthetic_tafsir_sources: tuple = ("tafsir_muyassar", "tafsir_saadi")   # tafsir passages that also get questions
    synthetic_tafsir_sample: int = 6236                  # per tafsir source (they are long; keep the cost bounded)
    synthetic_hadith_sample: int = 12000                 # hadith that get generated questions
    hadith_chapter_holdout: float = 0.2                  # share of hadith chapters (with a title and >= 5 hadith) kept for testing only
    hadith_per_chapter: int = 5                          # title -> hadith pairs per chapter (big chapters must not dominate)
    dedupe_overlap: float = 0.6                          # drop a question sharing more than this fraction of its tokens with the passage
    roundtrip_top: int = 50                              # keep a question only if base FADA ranks its source passage this high
    neg_ranks: tuple = (10, 50)                          # hard negatives: base FADA ranks in this range
    neg_window: int = 2                                  # skip negatives within +-2 ayat of the positive (false negatives)
    replay_share: float = 0.2                            # Arab3M rows as a share of the final training set
    mrl_dims: tuple = (768, 512, 256, 128, 64)
    max_seq_length: int = 200
    batch_size: int = 256               # small data: more updates beat a big batch (hard negatives still come with every row)
    mini_batch_size: int = 128
    scale: float = 20.0
    learning_rate: float = 5e-6
    weight_decay: float = 0.1
    epochs: int = 6
    save_steps: int = 300
    seed: int = 42
    out: str = "/content/fada-quran"
    dry_run: bool = False
    wandb_project: str = "fada-quran"
    run_name: str = "fada-quran-ft"

cfg = Cfg()
os.makedirs(cfg.out, exist_ok=True)
random.seed(cfg.seed); np.random.seed(cfg.seed)

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

if secret("HF_TOKEN"):
    from huggingface_hub import login
    login(secret("HF_TOKEN"))

import wandb
wandb.login(key=secret("WANDB_API_KEY")) if secret("WANDB_API_KEY") else wandb.login()
os.environ["WANDB_LOG_MODEL"] = "false"
run = wandb.init(project=cfg.wandb_project, name=cfg.run_name, job_type="finetune", config=asdict(cfg))

from sentence_transformers import SentenceTransformer

def fada_path(src):
    kind, _, ref = src.partition(":")
    if kind == "path":
        return ref
    if kind == "hf":
        from huggingface_hub import snapshot_download
        return snapshot_download(ref)
    if kind == "wandb":
        return run.use_artifact(ref, type="model").download()
    raise ValueError("fada_source must start with hf:, wandb: or path:")

fada_src = cfg.fada_source or f"wandb:{run.entity}/{cfg.fada_artifact}"
FADA_DIR = fada_path(fada_src)
base = SentenceTransformer(FADA_DIR, device="cuda")
base.max_seq_length = cfg.max_seq_length
assert base.get_embedding_dimension() == max(cfg.mrl_dims), "FADA size does not match cfg.mrl_dims"
print("FADA loaded from:", fada_src, "->", FADA_DIR)

In [ ]:
# corpus (from your database): original `text` is never changed; `encoding` is what production embeds (diacritics removed)
if not os.path.exists(cfg.corpus_path):          # pandas would treat a missing path as literal JSON and fail with "Expected object or value"
    try:
        from google.colab import files
        print("corpus.jsonl not found: choose the file made by scripts/quran_pairs.py")
        uploaded = files.upload()
        os.replace(next(iter(uploaded)), cfg.corpus_path)
    except ImportError:
        raise FileNotFoundError(f"{cfg.corpus_path} not found: set cfg.corpus_path to your corpus.jsonl")
corpus = pd.read_json(cfg.corpus_path, lines=True)
assert len(corpus) and {"source", "surah", "ayah", "encoding"} <= set(corpus.columns), "not a corpus.jsonl from scripts/quran_pairs.py"
if "chapter_title" not in corpus:
    corpus["chapter_title"] = ""
corpus["chapter_title"] = corpus.chapter_title.fillna("")
sources = sorted(corpus.source.unique())
HAS_HADITH = "hadith" in sources
print("hadith included:", HAS_HADITH)
print(corpus.groupby("source").size().to_string())

# self-test: the notebook must embed the same text production does. `encoding` is the production cleanup output; we only
# check it is not empty and has no vowel marks, so a corpus exported without that column fails here
DIACRITICS = re.compile("[\u064b-\u0652]")      # the vowel marks production removes; Quranic annotation marks (madda, small signs) stay on purpose
assert (corpus.encoding.str.len() > 0).all(), "empty encoding column"
assert not corpus.encoding.str.contains(DIACRITICS).any(), "encoding still has diacritics: export with scripts/quran_pairs.py"

In [ ]:
# held-out test set: QRCD (CC BY-ND: evaluation only). pq_id looks like "9:60-61_316" -> surah 9, ayat 60..61 are the gold passage
from datasets import load_dataset
qrcd = load_dataset("Raniahossam33/ARABIC_QURAN_DATASET", split="train").to_pandas()
PQ = re.compile(r"^(\d+):(\d+)(?:-(\d+))?_")
def gold_of(pq_id):
    m = PQ.match(pq_id)
    s, a, b = int(m[1]), int(m[2]), int(m[3] or m[2])
    return [(s, i) for i in range(a, b + 1)]
qrcd["gold"] = qrcd.pq_id.map(gold_of)
eval_rows = [{"question": q, "gold": [list(g) for g in gl]} for q, gl in zip(qrcd.question, qrcd.gold)]
HOLDOUT = {g for gl in qrcd.gold for g in gl}
with open(os.path.join(cfg.out, "eval.jsonl"), "w", encoding="utf-8") as f:       # same file the repo's scripts/quran_eval.py reads
    f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in eval_rows)
print(f"{len(eval_rows)} held-out questions, {len(HOLDOUT)} held-out ayat (removed from all training data)")

is_hadith = corpus.source == "hadith"
corpus["held_out"] = [(not h) and (s, a) in HOLDOUT for h, s, a in zip(is_hadith, corpus.surah, corpus.ayah)]

# held-out hadith chapters: no labels exist, so whole chapters are the test. query = the chapter title, gold = its hadith
hadith_eval_rows, HOLDOUT_CHAPTERS = [], set()
if HAS_HADITH:
    had = corpus[is_hadith]
    sizes = had.groupby("surah").size()
    titled = had[had.chapter_title.str.len() > 3].groupby("surah").chapter_title.first()
    eligible = sorted(c for c in titled.index if sizes[c] >= 5)
    rng = random.Random(cfg.seed)
    HOLDOUT_CHAPTERS = set(rng.sample(eligible, int(len(eligible) * cfg.hadith_chapter_holdout)))
    for c in sorted(HOLDOUT_CHAPTERS):
        gold = [[int(c), int(i)] for i in had[had.surah == c].ayah]
        hadith_eval_rows.append({"question": titled[c], "gold": gold})
    corpus.loc[is_hadith & corpus.surah.isin(HOLDOUT_CHAPTERS), "held_out"] = True
    with open(os.path.join(cfg.out, "hadith_eval.jsonl"), "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in hadith_eval_rows)
    print(f"{len(hadith_eval_rows)} held-out hadith chapters (titles are the queries), {int(corpus.held_out[is_hadith].sum())} hadith removed from training")
train_corpus = corpus[~corpus.held_out].reset_index(drop=True)

In [ ]:
# base-model recall on the held-out set (the number to beat). Ayah-level units of each source, cosine, no keyword step
def embed(model, texts, dim=None, batch=512):
    e = model.encode(texts, batch_size=batch, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    return e[:, :dim] if dim else e

# (eval rows, sources they are scored against, capped). QRCD: recall = any gold ayah in the top k. Hadith chapters have
# 5-600 gold hadith, so the score is hits in the top k divided by min(k, number of gold) (the same "capped recall" as scripts/recall_proxy.py)
EVAL_SETS = {"qrcd": (eval_rows, ("quran", "tafsir_muyassar", "tafsir_saadi"), False)}
if HAS_HADITH and hadith_eval_rows:
    EVAL_SETS["hadith-chapters"] = (hadith_eval_rows, ("hadith",), True)

def recall_table(model, label, dims=(768, 256, 64), ks=(1, 3, 8), only=None):
    rows = {}
    for set_name, (qrows, srcs, capped) in EVAL_SETS.items():
        full_q = embed(model, [r["question"] for r in qrows])     # no prefix, same as production
        golds = [set(map(tuple, r["gold"])) for r in qrows]
        for src in [s for s in srcs if s in sources and (only is None or s in only)]:
            part = corpus[corpus.source == src].reset_index(drop=True)
            refs = list(zip(part.surah, part.ayah))
            full_d = embed(model, part.encoding.tolist())
            for d in dims:
                dm = full_d[:, :d] / np.linalg.norm(full_d[:, :d], axis=1, keepdims=True)
                qm = full_q[:, :d] / np.linalg.norm(full_q[:, :d], axis=1, keepdims=True)
                order = np.argsort(-(qm @ dm.T), axis=1)[:, : max(ks)]
                res = {f"recall@{k}": [] for k in ks}; rr = []
                for row, gold in zip(order, golds):
                    hit = [refs[j] in gold for j in row]
                    first = next((i + 1 for i, h in enumerate(hit) if h), None)
                    rr.append(1 / first if first else 0)
                    for k in ks:
                        res[f"recall@{k}"].append(sum(hit[:k]) / min(k, len(gold)) if capped else float(any(hit[:k])))
                out = {k: np.mean(v) * 100 for k, v in res.items()}
                out["mrr"] = np.mean(rr) * 100
                rows[f"{label} | {src} | {d}d"] = out
    return pd.DataFrame(rows).T.round(2)

before = recall_table(base, "FADA before")
print(before.to_string())
run.log({"benchmark/quran_before": wandb.Table(dataframe=before.reset_index().rename(columns={"index": "run"}))})

In [ ]:
# training pairs 1: AQQAC (CC BY 4.0). chapter_no + verse_list give the gold ayat; positives are those ayat's texts
aq = load_dataset("nazimali/quran-question-answer-context", split="train").to_pandas()
by_ref = {(r.source, r.surah, r.ayah): r for r in corpus.itertuples()}
pairs = []        # dicts: anchor, positive, pos_source, pos_ref (surah, ayah), kind
for r in aq.itertuples():
    verses = json.loads(r.verse_list) if isinstance(r.verse_list, str) else list(r.verse_list)
    for v in verses:
        ref = (int(r.chapter_no), int(v))
        if ref in HOLDOUT:
            continue
        for src in ("quran", "tafsir_muyassar"):
            row = by_ref.get((src, *ref))
            if row is not None:
                pairs.append(dict(anchor=r.question.strip(), positive=row.encoding, pos_source=src, pos_ref=ref, kind="aqqac"))
print("AQQAC pairs:", len(pairs))

# training pairs 2: ayah <-> its tafsir (both directions)
quran_by_ref = {(r.surah, r.ayah): r.encoding for r in train_corpus[train_corpus.source == "quran"].itertuples()}
for r in train_corpus[train_corpus.source != "quran"].itertuples():
    q = quran_by_ref.get((r.surah, r.ayah))
    if q:
        pairs.append(dict(anchor=q, positive=r.encoding, pos_source=r.source, pos_ref=(r.surah, r.ayah), kind="ayah2tafsir"))
        pairs.append(dict(anchor=r.encoding, positive=q, pos_source="quran", pos_ref=(r.surah, r.ayah), kind="tafsir2ayah"))
print("with tafsir pairs:", len(pairs))

# training pairs 2b: hadith chapter title -> hadith matn (a few hadith per chapter, held-out chapters already excluded)
if HAS_HADITH:
    n0 = len(pairs)
    for c, part in train_corpus[(train_corpus.source == "hadith") & (train_corpus.chapter_title.str.len() > 3)].groupby("surah"):
        for r in part.sample(min(cfg.hadith_per_chapter, len(part)), random_state=cfg.seed).itertuples():
            pairs.append(dict(anchor=r.chapter_title.strip(), positive=r.encoding, pos_source="hadith", pos_ref=(r.surah, r.ayah), kind="chapter_title"))
    print("hadith title pairs:", len(pairs) - n0)

In [ ]:
# training pairs 3: synthetic questions written by an open model on this GPU (resumable)
from vllm import LLM, SamplingParams
llm = LLM(cfg.generator, dtype="bfloat16", max_model_len=4096, gpu_memory_utilization=0.85, seed=cfg.seed)
tok = llm.get_tokenizer()

PROMPT = (
    "أنت تكتب أسئلة باللغة العربية الفصحى يطرحها مستخدم عادي على محرك بحث عن القرآن وتفسيره والحديث النبوي.\n"
    "النص:\n{text}\n\n"
    "اكتب {n} أسئلة مختلفة يمكن الإجابة عنها بهذا النص، بأساليب مختلفة: سؤال مباشر، سؤال عن الموضوع أو الحكم، "
    "وسؤال بكلمات مختلفة عن كلمات النص. لا تنسخ عبارات من النص ولا تذكر رقم الآية أو اسم السورة أو اسم الراوي أو الكتاب.\n"
    "أجب بقائمة JSON فقط من النصوص، بلا شرح."
)

def chat(text):
    msgs = [{"role": "user", "content": PROMPT.format(text=text[:900], n=cfg.questions_per_item)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def parse_questions(raw):
    m = re.search(r"\[.*\]", raw, re.S)
    try:
        items = json.loads(m.group(0)) if m else []
    except json.JSONDecodeError:
        return []
    return [q.strip() for q in items if isinstance(q, str) and len(q.strip()) > 8]

def tokens(s): return set(re.findall(r"\w+", s))

def keep_question(q, passage):
    t = tokens(q)
    return len(t) >= 2 and len(t & tokens(passage)) / len(t) <= cfg.dedupe_overlap

# one work list: (source, surah, ayah, passage text)
items = [(r.source, r.surah, r.ayah, r.encoding) for r in train_corpus[train_corpus.source.isin(cfg.synthetic_ayat_sources)].itertuples()]
for s in cfg.synthetic_tafsir_sources:
    part = train_corpus[train_corpus.source == s]
    items += [(r.source, r.surah, r.ayah, r.encoding) for r in part.sample(min(cfg.synthetic_tafsir_sample, len(part)), random_state=cfg.seed).itertuples()]
if HAS_HADITH:
    part = train_corpus[train_corpus.source == "hadith"]
    items += [(r.source, r.surah, r.ayah, r.encoding) for r in part.sample(min(cfg.synthetic_hadith_sample, len(part)), random_state=cfg.seed).itertuples()]
if cfg.dry_run:
    items = random.sample(items, 200)

cache = os.path.join(cfg.out, "synthetic.jsonl")
done = {}
if os.path.exists(cache):
    for line in open(cache, encoding="utf-8"):
        d = json.loads(line); done[(d["source"], d["surah"], d["ayah"])] = d
todo = [it for it in items if it[:3] not in done]
print(f"{len(done)} cached, {len(todo)} to generate with {cfg.generator}")

params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=300, seed=cfg.seed)
with open(cache, "a", encoding="utf-8") as f:
    for i in range(0, len(todo), 2048):
        chunk = todo[i : i + 2048]
        outs = llm.generate([chat(t[3]) for t in chunk], params)
        for (src, s, a, text), o in zip(chunk, outs):
            qs = [q for q in parse_questions(o.outputs[0].text) if keep_question(q, text)]
            f.write(json.dumps(dict(source=src, surah=s, ayah=a, questions=qs), ensure_ascii=False) + "\n")
        f.flush(); print(f"{min(i + 2048, len(todo))}/{len(todo)}", flush=True)

synthetic = [json.loads(l) for l in open(cache, encoding="utf-8")]
text_of = {(r.source, r.surah, r.ayah): r.encoding for r in train_corpus.itertuples()}
n0 = len(pairs)
for d in synthetic:
    for q in d["questions"]:
        pairs.append(dict(anchor=q, positive=text_of[(d["source"], d["surah"], d["ayah"])], pos_source=d["source"],
                          pos_ref=(d["surah"], d["ayah"]), kind="synthetic"))
print("synthetic pairs:", len(pairs) - n0)

# spot check by hand before trusting them: read these 40
for p in random.sample([p for p in pairs if p["kind"] == "synthetic"], 40):
    print(f'{p["pos_source"]} {p["pos_ref"]} | Q: {p["anchor"]}')
del llm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# round-trip filter + hard negatives, both with BASE FADA. Passage index per source = ayah-level texts (no held-out ayat)
index = {}
for src in sources:
    part = train_corpus[train_corpus.source == src].reset_index(drop=True)
    index[src] = dict(refs=list(zip(part.surah, part.ayah)), texts=part.encoding.tolist(),
                      emb=torch.tensor(embed(base, part.encoding.tolist())).cuda())
    print(src, len(part))

kept = []
by_src = {}
for p in pairs:
    by_src.setdefault(p["pos_source"], []).append(p)
for src, plist in by_src.items():
    ix = index[src]
    q = torch.tensor(embed(base, [p["anchor"] for p in plist])).cuda()
    top = torch.topk(q @ ix["emb"].T, k=min(cfg.neg_ranks[1], len(ix["refs"])), dim=1).indices.cpu().numpy()
    pos_of = {r: i for i, r in enumerate(ix["refs"])}
    for p, row in zip(plist, top):
        pos_i = pos_of.get(tuple(p["pos_ref"]))
        if pos_i is None:
            continue
        if p["kind"] == "synthetic" and pos_i not in row[: cfg.roundtrip_top]:
            continue                                   # noise filter: the question does not even land near its source
        s0, a0 = p["pos_ref"]
        for j in row[cfg.neg_ranks[0] : cfg.neg_ranks[1]]:
            s1, a1 = ix["refs"][j]
            if s1 == s0 and (src == "hadith" or abs(a1 - a0) <= cfg.neg_window):
                continue                               # neighbours (and a hadith's own chapter) may be right answers too
            kept.append(dict(anchor=p["anchor"], positive=p["positive"], negative=ix["texts"][j], kind=p["kind"]))
            break
print(f"{len(pairs)} pairs -> {len(kept)} triplets with a hard negative")
del index; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# replay (general Arabic) so the model does not forget it, then the final training set + leak check
from datasets import Dataset
COLS = ["anchor", "positive", "negative"]
quran_df = pd.DataFrame(kept)[COLS + ["kind"]].drop_duplicates(subset=COLS)
n_replay = int(len(quran_df) * cfg.replay_share / (1 - cfg.replay_share))
rep = load_dataset("Omartificial-Intelligence-Space/Arab3M-Triplets", split="train").select_columns(COLS).shuffle(seed=cfg.seed).select(range(n_replay)).to_pandas()
rep["kind"] = "replay"
df = pd.concat([quran_df, rep], ignore_index=True)
for c in COLS:
    df[c] = df[c].astype("string").str.strip()
df = df.dropna()
df = df[(df[COLS] != "").all(axis=1)]
df = df[(df.anchor != df.positive) & (df.positive != df.negative) & (df.anchor != df.negative)].reset_index(drop=True)
if cfg.dry_run:
    df = df.sample(min(len(df), 2000), random_state=cfg.seed).reset_index(drop=True)

# no held-out text may be a training positive or negative
held_text = set(corpus[corpus.held_out].encoding)
n_before = len(df)
df = df[~df.positive.isin(held_text) & ~df.negative.isin(held_text)].reset_index(drop=True)   # a hadith or verse repeated elsewhere counts as held out too
print(f"dropped {n_before - len(df)} rows whose text equals a held-out text")
assert not (set(df.positive) | set(df.negative)) & held_text, "held-out text leaked into the training set"
print(df.kind.value_counts().to_string())
json.dump({"holdout": sorted(map(list, HOLDOUT)), "counts": df.kind.value_counts().to_dict()}, open(os.path.join(cfg.out, "splits.json"), "w"))
run.summary.update({"train_rows": len(df), "heldout_questions": len(eval_rows)})
train_ds = Dataset.from_pandas(df[COLS], preserve_index=False)

In [ ]:
# model, loss, optimizer, trainer (same recipe as the ARBERTv2 notebook: Matryoshka + Cached MNRL, Lion)
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from sentence_transformers.sentence_transformer.losses import MatryoshkaLoss, CachedMultipleNegativesRankingLoss
from transformers.trainer_utils import get_last_checkpoint
from lion_pytorch import Lion

model = SentenceTransformer(FADA_DIR, model_kwargs={"attn_implementation": "sdpa"}, device="cuda")
model.max_seq_length = cfg.max_seq_length
loss = MatryoshkaLoss(model, CachedMultipleNegativesRankingLoss(model, scale=cfg.scale, mini_batch_size=cfg.mini_batch_size), list(cfg.mrl_dims))

def decays(n, p): return p.ndim >= 2 and "norm" not in n.lower()
optimizer = Lion(
    [{"params": [p for n, p in model.named_parameters() if decays(n, p)], "weight_decay": cfg.weight_decay},
     {"params": [p for n, p in model.named_parameters() if not decays(n, p)], "weight_decay": 0.0}],
    lr=cfg.learning_rate, betas=(0.9, 0.99), use_triton=False)

class SortedCollator(SentenceTransformerTrainer.data_collator_class):
    def __call__(self, features):
        if features and "anchor" in features[0]:
            features = sorted(features, key=lambda r: len(r["anchor"]))
        return super().__call__(features)

class SortedTrainer(SentenceTransformerTrainer):
    data_collator_class = SortedCollator

ckpt_dir, final_dir = os.path.join(cfg.out, "checkpoints"), os.path.join(cfg.out, "final")
steps_per_epoch = max(1, len(train_ds) // cfg.batch_size)
max_steps = 50 if cfg.dry_run else steps_per_epoch * cfg.epochs
args = SentenceTransformerTrainingArguments(
    output_dir=ckpt_dir, per_device_train_batch_size=cfg.batch_size, max_steps=max_steps,
    learning_rate=cfg.learning_rate, lr_scheduler_type="cosine", warmup_steps=0.1, max_grad_norm=1.0, bf16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES_HASHED, dataloader_num_workers=min((os.cpu_count() or 4) - 1, 16),
    dataloader_drop_last=True, save_steps=cfg.save_steps, logging_steps=10,
    report_to="wandb", run_name=cfg.run_name, seed=cfg.seed, data_seed=cfg.seed)
trainer = SortedTrainer(model=model, args=args, train_dataset=train_ds, loss=loss, optimizers=(optimizer, None))

from transformers import TrainerCallback
class CkptEval(TrainerCallback):
    """At every checkpoint: recall@8 at 64 dims on the held-out sets, so a weak run shows early."""
    def on_save(self, args, state, control, model=None, **kw):
        t = recall_table(model, f"step {state.global_step}", dims=(64,), ks=(8,), only=("quran", "hadith"))
        print(t.to_string())
        run.log({f"ckpt/{k}": v["recall@8"] for k, v in t.iterrows()}, step=state.global_step)
trainer.add_callback(CkptEval())
print(f"{len(train_ds):,} rows, {max_steps} steps of {cfg.batch_size}")

In [ ]:
# train (resumes from the latest checkpoint)
trainer.train(resume_from_checkpoint=get_last_checkpoint(ckpt_dir) if os.path.isdir(ckpt_dir) else None)
trainer.save_model(final_dir)
art = wandb.Artifact("fada-quran", type="model", metadata={"base": fada_src, "rows": len(train_ds), "generator": cfg.generator})
art.add_dir(final_dir)
run.log_artifact(art)
print("saved:", final_dir)

In [ ]:
# result: Quran recall before vs after, then the general-Arabic check (STS) so we know nothing broke
del model, trainer; gc.collect(); torch.cuda.empty_cache()
tuned = SentenceTransformer(final_dir, device="cuda"); tuned.max_seq_length = cfg.max_seq_length
after = recall_table(tuned, "FADA tuned")
both = pd.concat([before, after])
print(both.to_string())
run.log({"benchmark/quran_after": wandb.Table(dataframe=both.reset_index().rename(columns={"index": "run"}))})

import mteb
tasks = [mteb.get_task("STS17", hf_subsets=["ar-ar"]), mteb.get_task("STS22.v2", hf_subsets=["ar"])]
def sts(path, label, dim):
    m = SentenceTransformer(path, device="cuda", truncate_dim=dim); m.max_seq_length = 512
    res = mteb.evaluate(m, tasks, cache=None, overwrite_strategy="always", encode_kwargs={"batch_size": 256}, show_progress_bar=False)
    return {label: {r.task_name: r.get_score() * 100 for r in res.task_results}}
rows = {}
for d in (768, 64):
    rows.update(sts(FADA_DIR, f"FADA before, {d}d", d)); rows.update(sts(final_dir, f"FADA tuned, {d}d", d))
sts_df = pd.DataFrame(rows).T; sts_df["average"] = sts_df.mean(axis=1)
print(sts_df.round(2).to_string())
run.log({"benchmark/sts": wandb.Table(dataframe=sts_df.round(2).reset_index().rename(columns={"index": "run"}))})

# ship gate from the plan: recall@8 at 64 dims up >= 15 points on the Quran source, STS average down <= 1.5
q8 = after.loc["FADA tuned | quran | 64d", "recall@8"] - before.loc["FADA before | quran | 64d", "recall@8"]
d_sts = sts_df.loc["FADA tuned, 64d", "average"] - sts_df.loc["FADA before, 64d", "average"]
h8 = None
if "hadith-chapters" in EVAL_SETS:
    h8 = after.loc["FADA tuned | hadith | 64d", "recall@8"] - before.loc["FADA before | hadith | 64d", "recall@8"]
print(f"quran recall@8 (64d) {q8:+.1f} points | hadith chapter capped recall@8 (64d) {h8 if h8 is None else round(h8, 1)} | STS average (64d) {d_sts:+.2f}")
ok = q8 >= 15 and d_sts >= -1.5 and (h8 is None or h8 >= 5)       # the proxy baseline for hadith chapters is only ~6%, so +5 points is already large
print("GATE:", "PASS" if ok else "FAIL: see the plan (separate Quran encoder, more data, or check the hadith rows)")
wandb.finish()

In [ ]:
# download: the model folder, the held-out eval file (for scripts/quran_eval.py on your machine) and the split record
!cd {cfg.out} && zip -qr fada-quran-final.zip final eval.jsonl hadith_eval.jsonl splits.json
from google.colab import files
files.download(os.path.join(cfg.out, "fada-quran-final.zip"))